# Phase-by-Phase Model Probe

Each scenario cell below builds one controlled traffic situation, feeds the corresponding 10-value state vector to the trained DQN, and displays both the lane-level queue table and the model output.

The model only receives aggregate approach queues and wait times, so the left/straight/right lane counts are shown for human interpretation and then summed into `A/B/C/D` queues before inference.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from State.traffic_config import PHASE_DURATION_CHOICES_SECONDS, STATE_NORMALIZATION
from State.traffic_env import TrafficEnv
from Neural_Networks.DQN_Implementation.dqn import DQN

env = TrafficEnv()
PHASE_NAMES = env.phases
ACTIONS = env.actions
APPROACHES = ["A", "B", "C", "D"]
TURNS = env.turns
state_dim = len(env._get_state())
action_dim = len(ACTIONS)

model = DQN(state_dim, action_dim)
model_path = project_root / "code" / "Neural_Networks" / "DQN_Implementation" / "traffic_dqn_model1000.pth"
checkpoint = torch.load(model_path, map_location=torch.device("cpu"))
if "model_state_dict" not in checkpoint:
    raise RuntimeError("This is a legacy checkpoint without environment metadata. Rerun dqn.ipynb to retrain and resave the model.")
if checkpoint["state_dim"] != state_dim or checkpoint["action_dim"] != action_dim or checkpoint["phases"] != PHASE_NAMES:
    raise RuntimeError(
        "Checkpoint metadata does not match the current environment. "
        "Rerun dqn.ipynb to retrain and resave the model."
    )
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
def lane_wait_value(waits, approach, turn):
    value = waits.get(approach, 0)
    if isinstance(value, dict):
        return value.get(turn, 0)
    return value


def make_state(lanes, waits, current_phase=0, time_in_phase=0, last_duration=10):
    lane_queues = np.array(
        [lanes[a].get(turn, 0) for a in APPROACHES for turn in TURNS],
        dtype=float,
    )
    lane_waits = np.array(
        [lane_wait_value(waits, a, turn) for a in APPROACHES for turn in TURNS],
        dtype=float,
    )
    return np.concatenate([
        lane_queues / STATE_NORMALIZATION["queue"],
        lane_waits / STATE_NORMALIZATION["wait"],
        [current_phase / len(PHASE_NAMES)],
        [time_in_phase / STATE_NORMALIZATION["phase_time"]],
        [last_duration / max(PHASE_DURATION_CHOICES_SECONDS)],
    ])


def predict(state):
    with torch.no_grad():
        q_values = model(torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)).squeeze(0).numpy()
    action = int(np.argmax(q_values))
    return action, q_values


def decode_action(action_index):
    action = ACTIONS[action_index]
    return action["phase_index"], action["duration_seconds"], action["label"]


def scenario_table(lanes, waits, current_phase, time_in_phase):
    rows = []
    for approach in APPROACHES:
        lane_counts = lanes[approach]
        rows.append({
            "approach": approach,
            "left_queue": lane_counts.get("left", 0),
            "straight_queue": lane_counts.get("straight", 0),
            "right_queue": lane_counts.get("right", 0),
            "total_queue": sum(lane_counts.values()),
            "left_wait_s": lane_wait_value(waits, approach, "left"),
            "straight_wait_s": lane_wait_value(waits, approach, "straight"),
            "right_wait_s": lane_wait_value(waits, approach, "right"),
            "current_phase": PHASE_NAMES[current_phase],
            "time_in_phase_s": time_in_phase,
        })
    return pd.DataFrame(rows)


def q_table(q_values, action):
    rows = []
    for i, value in enumerate(q_values):
        phase_index, duration, label = decode_action(i)
        rows.append({
            "action_id": i,
            "phase": PHASE_NAMES[phase_index],
            "duration_s": duration,
            "q_value": value,
            "selected": i == action,
        })
    return pd.DataFrame(rows).sort_values("q_value", ascending=False)


def run_scenario(name, lanes, waits=None, current_phase=0, time_in_phase=0, last_duration=10):
    waits = waits or {a: 0 for a in APPROACHES}
    state = make_state(lanes, waits, current_phase, time_in_phase, last_duration)
    action, q_values = predict(state)
    phase_index, duration, label = decode_action(action)
    display(Markdown(f"## {name}"))
    display(Markdown(f"**Selected action:** `{action}` / `{PHASE_NAMES[phase_index]}` for `{duration}s`"))
    display(Markdown("**Lane-level table used by the model:**"))
    display(scenario_table(lanes, waits, current_phase, time_in_phase))
    display(Markdown("**Normalized model input:**"))
    columns = [f"{a}_{turn}_queue" for a in APPROACHES for turn in TURNS]
    columns += [f"{a}_{turn}_wait" for a in APPROACHES for turn in TURNS]
    columns += ["current_phase", "time_in_phase", "last_duration"]
    display(pd.DataFrame([state], columns=columns))
    display(Markdown("**Top model output Q-values:**"))
    display(q_table(q_values, action).head(10))
    return action, q_values


In [ ]:
# Situation 1: balanced light traffic on all approaches.
balanced_lanes = {
    "A": {"left": 1, "straight": 2, "right": 1},
    "B": {"left": 1, "straight": 2, "right": 1},
    "C": {"left": 1, "straight": 2, "right": 1},
    "D": {"left": 1, "straight": 2, "right": 1},
}
balanced_waits = {"A": 6, "B": 6, "C": 6, "D": 6}
run_scenario("Situation 1: balanced queues", balanced_lanes, balanced_waits, current_phase=0, time_in_phase=8)


In [ ]:
# Situation 2: A/C through traffic is heavy, so AC_forward should be attractive if the model learned demand service.
ac_through_lanes = {
    "A": {"left": 1, "straight": 9, "right": 3},
    "B": {"left": 0, "straight": 2, "right": 1},
    "C": {"left": 2, "straight": 8, "right": 2},
    "D": {"left": 1, "straight": 1, "right": 1},
}
ac_through_waits = {"A": 18, "B": 4, "C": 16, "D": 5}
run_scenario("Situation 2: heavy A/C straight and right demand", ac_through_lanes, ac_through_waits, current_phase=1, time_in_phase=20)


In [ ]:
# Situation 3: B/D through traffic dominates.
bd_through_lanes = {
    "A": {"left": 1, "straight": 1, "right": 0},
    "B": {"left": 2, "straight": 10, "right": 3},
    "C": {"left": 0, "straight": 2, "right": 1},
    "D": {"left": 2, "straight": 9, "right": 2},
}
bd_through_waits = {"A": 3, "B": 21, "C": 5, "D": 19}
run_scenario("Situation 3: heavy B/D straight and right demand", bd_through_lanes, bd_through_waits, current_phase=0, time_in_phase=18)


In [ ]:
# Situation 4: A/C left-turn pockets are backing up.
ac_left_lanes = {
    "A": {"left": 8, "straight": 2, "right": 1},
    "B": {"left": 1, "straight": 2, "right": 1},
    "C": {"left": 7, "straight": 1, "right": 1},
    "D": {"left": 1, "straight": 2, "right": 0},
}
ac_left_waits = {"A": 24, "B": 7, "C": 23, "D": 8}
run_scenario("Situation 4: protected A/C left-turn pressure", ac_left_lanes, ac_left_waits, current_phase=0, time_in_phase=25)


In [ ]:
# Situation 5: B/D left-turn pockets are backing up.
bd_left_lanes = {
    "A": {"left": 1, "straight": 2, "right": 1},
    "B": {"left": 9, "straight": 1, "right": 1},
    "C": {"left": 1, "straight": 2, "right": 1},
    "D": {"left": 8, "straight": 2, "right": 1},
}
bd_left_waits = {"A": 6, "B": 28, "C": 7, "D": 26}
run_scenario("Situation 5: protected B/D left-turn pressure", bd_left_lanes, bd_left_waits, current_phase=1, time_in_phase=24)


In [ ]:
# Situation 6: starvation check. A modest B/D queue has waited much longer than larger A/C traffic.
starvation_lanes = {
    "A": {"left": 2, "straight": 6, "right": 2},
    "B": {"left": 1, "straight": 3, "right": 1},
    "C": {"left": 2, "straight": 6, "right": 2},
    "D": {"left": 1, "straight": 3, "right": 1},
}
starvation_waits = {"A": 9, "B": 42, "C": 10, "D": 40}
run_scenario("Situation 6: B/D starvation pressure", starvation_lanes, starvation_waits, current_phase=0, time_in_phase=30)
